# Download fake videos and upload to HF

Sometimes modelscope block access for the Ukrainian IP, so Google Colab helps with this issue

In [1]:
!pip install -q huggingface_hub requests tqdm

In [2]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os, shutil, tarfile, random
from pathlib import Path
import requests
from tqdm import tqdm
from huggingface_hub import upload_folder

In [ ]:
REPO_ID = 'ironiss/PhysVidDetect-v1'
MODELSCOPE_PATH = 'https://www.modelscope.cn/api/v1/datasets/cccnju/GenVideo-100K/repo?Source=SDK&Revision=master&FilePath={}&View=False'
VIDEO_EXTS = {'.mp4', '.mov', '.webm', '.mkv', '.avi'}
SEED = 42

FAKE_SOURCES = {
    'ZeroScope':    'ZeroScope.tar.gz',
    'SVD':  'SVD.tar.gz',
    'VideoCrafter': 'VideoCrafter.tar.gz',
    'Pika': 'pika.tar.gz',
    'DynamicCrafter':   'DynamicCrafter.tar.gz',
    'SD':   'SD.tar.gz',
    'SEINE':    'SEINE.tar.gz',
    'Latte':    'Latte.tar.gz',
    'OpenSora': 'OpenSora.tar.gz',
}

WORK_DIR = Path('/content/fake_download')
WORK_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
import urllib3, warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings('ignore')

In [ ]:
for name, fn in FAKE_SOURCES.items():
    url = MODELSCOPE_PATH.format(fn)
    try:
        r = requests.head(url, timeout=30, verify=False, allow_redirects=True)
        size = r.headers.get('Content-Length', '?')
        size_gb = f"{int(size)/1e9:.2f} GB" if size != '?' else '?'
        print(f"{name:18} {r.status_code}  {size_gb}  ({fn})")
    except Exception as e:
        print(f"{name:18} ERROR  {e}")


ZeroScope          200  2.40 GB  (ZeroScope.tar.gz)
SVD                200  6.42 GB  (SVD.tar.gz)
VideoCrafter       200  15.44 GB  (VideoCrafter.tar.gz)
Pika               200  5.94 GB  (pika.tar.gz)
DynamicCrafter     200  16.62 GB  (DynamicCrafter.tar.gz)
SD                 200  16.12 GB  (SD.tar.gz)
SEINE              200  3.65 GB  (SEINE.tar.gz)
Latte              200  3.91 GB  (Latte.tar.gz)
OpenSora           200  1.79 GB  (OpenSora.tar.gz)


## Pick which sources to download

In [7]:
SOURCES_TO_DOWNLOAD = [list(FAKE_SOURCES)]
TARGET_PER_SOURCE = 3000

In [8]:
def download_archive(url, dest):
    with requests.get(url, stream=True, timeout=300, verify=False) as r:
        r.raise_for_status()
        total = int(r.headers.get('Content-Length', 0))
        with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc=dest.name) as bar:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))


def find_videos(root):
    out = []
    for r, _, files in os.walk(root):
        for f in files:
            if Path(f).suffix.lower() in VIDEO_EXTS:
                out.append(Path(r) / f)
    return out

In [ ]:

def process_source(name, archive_name):
    archive = WORK_DIR / archive_name
    extract_dir = WORK_DIR / 'extracted' / name
    extract_dir.mkdir(parents=True, exist_ok=True)
    out_dir = WORK_DIR / 'upload' / name
    out_dir.mkdir(parents=True, exist_ok=True)

    url = MODELSCOPE_PATH.format(archive_name)
    print(f'Started download {archive_name}')
    download_archive(url, archive)

    with tarfile.open(archive, 'r:*') as tf:
        tf.extractall(extract_dir)
    archive.unlink(missing_ok=True)

    videos = find_videos(extract_dir)

    if len(videos) > TARGET_PER_SOURCE:
        random.seed(SEED)
        videos = sorted(random.sample(videos, TARGET_PER_SOURCE))

    for v in tqdm(videos, desc='copying'):
        dst = out_dir / v.name
        if dst.exists():
            dst = out_dir / f'{v.parent.name}_{v.name}'
        shutil.copy2(v, dst)

    shutil.rmtree(extract_dir, ignore_errors=True)

    print(f'Upload videos to fake/{name}/ on HF')
    upload_folder(
        folder_path=str(out_dir),
        path_in_repo=f'fake/{name}',
        repo_id=REPO_ID,
        repo_type='dataset',
    )

    shutil.rmtree(out_dir, ignore_errors=True)
    print(f'{name} done')

In [ ]:
for src in SOURCES_TO_DOWNLOAD:
    print(f'Processing source: {src}')
    if src not in FAKE_SOURCES:
        continue
    process_source(src, FAKE_SOURCES[src])

shutil.rmtree(WORK_DIR, ignore_errors=True)